# Naive Bayes — Program

This notebook contains the data workflow and runnable code. See [Naive Bayes — Theory](07_naive_bayes_theory.ipynb) for concepts.

**Task:** Classification. **Data:** Wine recognition dataset. Run top to bottom.

## 1. Load data

In [ ]:
from sklearn.datasets import load_wine
data = load_wine(as_frame=True)
X = data.data.copy()
y = data.target.copy()

## 2. Inspect raw data

In [ ]:
print("Feature shape:", X.shape)
print("Types:\n", X.dtypes)
print("Missing by column:\n", X.isna().sum())
print("Duplicate feature rows:", X.duplicated().sum())
print("Preview:\n", X.head())
print("Target counts:\n", y.value_counts().sort_index())


## 3. Clean data
Drop duplicate observations and fill missing numeric values with each column median. The demo datasets are clean, but these checks and steps are useful for real data.

In [ ]:
import pandas as pd
if y is not None:
    combined = X.copy()
    combined["__target__"] = y
    combined = combined.drop_duplicates().reset_index(drop=True)
    y = combined.pop("__target__")
    X = combined
else:
    X = X.drop_duplicates().reset_index(drop=True)
X = X.apply(lambda col: col.fillna(col.median()) if col.isna().any() else col)
print("Clean feature shape:", X.shape)
print("Missing remaining:", int(X.isna().sum().sum()))
print("Duplicates remaining:", int(X.duplicated().sum()))

## 4. Analyze data
Review summary statistics and feature distributions. Classification also shows target balance.

In [ ]:
import matplotlib.pyplot as plt
print(X.describe().T[["mean", "std", "min", "max"]])
X.hist(figsize=(12, 8), bins=20)
plt.tight_layout(); plt.show()
y.value_counts().sort_index().plot(kind="bar", title="Target class counts")
plt.xlabel("Class"); plt.ylabel("Count"); plt.show()


## 5. Prepare training and test data
Use a fixed seed for repeatability. Class labels are stratified. Scaling is included in pipelines when needed.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:",len(X_train),"Test:",len(X_test))

## 6. Run the algorithm
Fit and evaluate the model. Metrics describe this example split; they are not guarantees of future performance.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
model = GaussianNB()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
print(classification_report(y_test, predictions, target_names=data.target_names))

## 7. Inspect an example output

In [ ]:
print("Predicted:", model.predict(X_test.iloc[[0]])[0])
print("Actual:", y_test.iloc[0])